# NB02 — Model Training × folds (GPU)

Trains the binary gate (`y = ALTERED`). BCE + `pos_weight`, AMP, early-stopping by val AUPRC.

## ⚙️ Only parameter to change: `QUICK_TEST`
- `QUICK_TEST = True`  → **quick test**: 1 model (mobilenetv3), 1 fold, 2 epochs. Only to verify if the environment works.
- `QUICK_TEST = False` → **full experiment**: 5 models × 5 folds = 25 runs, 40 epochs + early-stop.

**Resilience:** saves `*_last.ckpt` (resume) and `*_best.ckpt`; completed runs are automatically skipped;
interrupted runs resume from the last epoch. If the process stops, just re-execute the loop cell.

**Large GPU (H100 / RTX PRO 6000 Blackwell):** config automatically detects VRAM.
With >8 GB, all models use full batch = 32 (without gradient accumulation).


In [5]:
# ============================================================
QUICK_TEST = False      # <<< CHANGE TO False TO RUN EVERYTHING
# ============================================================
import sys, os
from pathlib import Path

# Mount Drive (in case the session restarted)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass # Running locally

# Point to the correct project directory on Drive
# (The same path used in NB00)
ROOT = Path('/content/drive/MyDrive/A4-Triagem-Binaria')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Set the A4_IMAGES_DIR that you configured in NB00
os.environ['A4_IMAGES_DIR'] = '/content/Imgs' # Or the path indicated by NB00

from configs import config as C
from src import train, utils
import torch, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

print('ROOT =', ROOT)
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU: {props.name}')
    print(f'VRAM: {vram_gb:.1f} GB | Compute: {props.major}.{props.minor}')
    print(f'Multi-processors: {props.multi_processor_count}')
print(f'device: {utils.get_device()} | AMP: {C.AMP}')
print(f'NUM_WORKERS: {C.NUM_WORKERS}')
print(f'IMAGES_DIR: {C.IMAGES_DIR} | exists: {C.IMAGES_DIR.exists()}')
print()
print('Micro-batch per model (auto-detected by VRAM):')
for k, v in C.MICRO_BATCH.items():
    accum = max(1, C.BATCH_SIZE // v)
    print(f'  {k}: micro={v} x accum={accum} -> effective={v*accum}')

## 1. Define execution plan based on `QUICK_TEST`
In quick test, only uses mobilenetv3 fold 0 for 2 epochs (doesn't pollute the actual experiment).


In [7]:
if QUICK_TEST:
    MODELS = ['mobilenetv3']
    FOLDS  = [0]
    MAX_EPOCHS = 2
    RESUME = False
    print('>> QUICK TEST MODE: mobilenetv3, fold 0, 2 epochs')
else:
    MODELS = list(C.MODELS.keys())
    FOLDS  = list(range(C.N_FOLDS))
    MAX_EPOCHS = C.EPOCHS
    RESUME = True
    print(f'>> FULL MODE: {len(MODELS)} models x {len(FOLDS)} folds = {len(MODELS)*len(FOLDS)} runs, {MAX_EPOCHS} epochs + early-stop')

jobs = [(m, f) for m in MODELS for f in FOLDS]
done = [(m,f) for (m,f) in jobs if RESUME and utils.is_run_done(C.METRICS_DIR, m, f)]
pending = [(m,f) for (m,f) in jobs if not (RESUME and utils.is_run_done(C.METRICS_DIR, m, f))]
print(f'runs to execute: {len(pending)} | already completed: {len(done)}')


## 2. Training loop (shows global progress + ETA per run)

In [8]:
timer = utils.Timer()
for i, (m, f) in enumerate(tqdm(pending, desc='runs')):
    print(f'\n=== [{i+1}/{len(pending)}] {m} fold {f} === (elapsed {timer.hms()})')
    out = train.train_fold(m, f, resume=RESUME, max_epochs=MAX_EPOCHS)
    print(f'    -> test AUPRC {out["test"]["auprc"]:.4f} | AUROC {out["test"]["auroc"]:.4f}')
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved  = torch.cuda.memory_reserved(0) / 1e9
        print(f'    -> VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB')
        torch.cuda.empty_cache()
print(f'\nCOMPLETED in {timer.hms()}')


## 3. (Only in quick test) Check and clean smoke artifacts

In [9]:
if QUICK_TEST:
    print('Smoke OK. Cleaning artifacts...')
    for m in MODELS:
        for f in FOLDS:
            p = C.METRICS_DIR / f'{m}_fold{f}_metrics.json'
            if p.exists(): p.unlink()
            for suf in ['_last.ckpt', '_best.ckpt']:
                p = C.CKPT_DIR / f'{m}_fold{f}{suf}'
                if p.exists(): p.unlink()
            npz = C.PRED_DIR / f'{m}_fold{f}_preds.npz'
            if npz.exists(): npz.unlink()
            for lang in ['en', 'pt']:
                fig = C.FIG_TRAIN / f'train_{m}_fold{f}_{lang}.png'
                if fig.exists(): fig.unlink()
            log = C.LOGS_DIR / f'{m}_fold{f}.log'
            if log.exists(): log.unlink()
    print('Cleaned. Change QUICK_TEST = False and re-execute for actual training.')
else:
    print('full mode — nothing to clean.')


## 4. Consolidate test metrics (mean ± SD per model) — only in full mode

In [10]:
if not QUICK_TEST:
    rows = {}
    for m in C.MODELS.keys():
        au, ap = [], []
        for f in range(C.N_FOLDS):
            mf = C.METRICS_DIR / f'{m}_fold{f}_metrics.json'
            if mf.exists():
                d = json.load(open(mf, encoding='utf-8'))['test']
                au.append(d['auroc']); ap.append(d['auprc'])
        if au:
            rows[m] = {'n_folds': len(au),
                       'auroc_mean': float(np.mean(au)), 'auroc_sd': float(np.std(au)),
                       'auprc_mean': float(np.mean(ap)), 'auprc_sd': float(np.std(ap))}
    df = pd.DataFrame(rows).T.sort_values('auprc_mean', ascending=False)
    display(df.round(4))
    utils.save_json(rows, C.METRICS_DIR / 'nb02_train_summary.json')
    print('saved: results/metrics/nb02_train_summary.json')
    print('Next: NB03 (operation/calibration) and NB04 (gate safety).')
else:
    print('Smoke test OK -> change QUICK_TEST=False and run the full experiment.')
